<a href="https://colab.research.google.com/github/dilipkumari20/Zepto_capstone_project.github/blob/main/capstone_m1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Module 1 — Data Pipeline (/data_pipeline)
import sqlite3
import statistics
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

BASE_URL = "https://books.toscrape.com/"
FIXED_GBP_TO_INR = 105.50

BASE_DIR = Path.cwd().resolve().parent
DB_PATH = BASE_DIR / "books.db"


# ---------------------------------------------------------
# Scraping helpers
# ---------------------------------------------------------

def get_soup(url):
    """
    Download a webpage and return its BeautifulSoup object.
    """
    response = requests.get(
        url,
        timeout=15,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")


def get_category_urls():
    """
    Get category names and URLs from the Books to Scrape
    category sidebar.
    """

    soup = get_soup(BASE_URL)

    categories = {}

    sidebar = soup.select(".side_categories ul li ul li a")

    for link in sidebar:
        category_name = link.get_text(strip=True)
        category_url = BASE_URL + link["href"]

        categories[category_name] = category_url

    return categories


def scrape_category(category_name, category_url):
    """
    Scrape all books from one category, including pagination.
    """

    books = []

    current_url = category_url

    while current_url:

        soup = get_soup(current_url)

        product_links = soup.select("article.product_pod h3 a")

        for product_link in product_links:

            relative_url = product_link.get("href")



            # Build the correct absolute URL
            detail_url = urljoin(current_url, relative_url)

            detail_soup = get_soup(detail_url)

            # -----------------------------
            # Title
            # -----------------------------
            title_tag = detail_soup.select_one("div.product_main h1")
            title = title_tag.get_text(strip=True) if title_tag else None

            # -----------------------------
            # Price
            # -----------------------------
            price_tag = detail_soup.select_one(
                "div.product_main p.price_color"
            )

            price_text = (
                price_tag.get_text(strip=True)
                if price_tag
                else None
            )

            # -----------------------------
            # Rating
            # -----------------------------
            rating_tag = detail_soup.select_one(
                "div.product_main p.star-rating"
            )

            star_rating = None

            if rating_tag:
                classes = rating_tag.get("class", [])

                for rating in ["One", "Two", "Three", "Four", "Five"]:
                    if rating in classes:
                        star_rating = rating
                        break

            # -----------------------------
            # Availability
            # -----------------------------
            availability_tag = detail_soup.select_one(
                "div.product_main p.instock.availability"
            )

            availability = (
                availability_tag.get_text(" ", strip=True)
                if availability_tag
                else None
            )

            books.append(
                {
                    "title": title,
                    "price": price_text,
                    "star_rating": star_rating,
                    "availability": availability,
                    "category": category_name,
                }
            )

        # -----------------------------
        # Find next page
        # -----------------------------
        next_link = soup.select_one("li.next a")

        if next_link:

            next_href = next_link.get("href")

            current_url = current_url.rsplit("/", 1)[0] + "/" + next_href

        else:
            current_url = None

    return books


def scrape_books(min_categories=3, minimum_books=60):
    """
    Scrape at least three categories until at least 60 books
    are collected.

    Categories are selected automatically.
    """

    categories = get_category_urls()

    print(f"Found {len(categories)} categories.")

    selected_categories = list(categories.items())[:min_categories]

    all_books = []

    for category_name, category_url in selected_categories:

        print(f"Scraping category: {category_name}")

        category_books = scrape_category(
            category_name,
            category_url
        )

        print(
            f"  Collected {len(category_books)} books"
        )

        all_books.extend(category_books)

    # If first 3 categories do not provide enough books,
    # continue with additional categories.
    if len(all_books) < minimum_books:

        for category_name, category_url in list(categories.items())[
            min_categories:
        ]:

            if category_name in dict(selected_categories):
                continue

            print(f"Scraping additional category: {category_name}")

            category_books = scrape_category(
                category_name,
                category_url
            )

            all_books.extend(category_books)

            if len(all_books) >= minimum_books:
                break

    df = pd.DataFrame(all_books)

    return df


# ---------------------------------------------------------
# Cleaning
# ---------------------------------------------------------

def parse_price(value):
    """
    Convert £X.XX into float.
    Return None when parsing fails.
    """

    if value is None:
        return None

    try:
        cleaned = (
            str(value)
            .replace("£", "")
            .replace(",", "")
            .strip()
        )

        return float(cleaned)

    except (ValueError, TypeError):
        return None


def parse_rating(value):
    """
    Convert textual rating to integer.
    """

    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5,
    }

    if value is None:
        return None

    return rating_map.get(str(value).strip())


def parse_stock(value):
    """
    Convert availability text to boolean.
    """

    if value is None:
        return None

    text = str(value).lower()

    if "in stock" in text:
        return True

    if "out of stock" in text:
        return False

    return None


def clean_data(df):
    """
    Clean scraped fields and create required columns.
    """

    df = df.copy()

    # ---------------------------------
    # Price
    # ---------------------------------

    df["price_gbp"] = df["price"].apply(parse_price)

    # ---------------------------------
    # Rating
    # ---------------------------------

    df["rating"] = df["star_rating"].apply(parse_rating)

    # ---------------------------------
    # Availability
    # ---------------------------------

    df["in_stock"] = df["availability"].apply(parse_stock)

    # ---------------------------------
    # Numeric median imputation
    # ---------------------------------

    if df["price_gbp"].isna().any():

        median_price = df["price_gbp"].median()

        df["price_gbp"] = df["price_gbp"].fillna(
            median_price
        )

    if df["rating"].isna().any():

        median_rating = df["rating"].median()

        df["rating"] = df["rating"].fillna(
            round(median_rating)
        )

    # ---------------------------------
    # Boolean parsing failures
    # ---------------------------------
    #
    # Since in_stock is not numeric, unexpected values
    # are handled conservatively by dropping those rows.
    #

    df = df.dropna(
        subset=[
            "title",
            "category",
            "in_stock"
        ]
    )

    # Convert types explicitly
    df["price_gbp"] = df["price_gbp"].astype(float)

    df["rating"] = df["rating"].astype(int)

    df["in_stock"] = df["in_stock"].astype(bool)

    # ---------------------------------
    # Fixed project conversion
    # ---------------------------------

    df["price_inr"] = (
        df["price_gbp"] * FIXED_GBP_TO_INR
    ).round(2)

    # Keep only required columns
    df = df[
        [
            "title",
            "price_gbp",
            "price_inr",
            "rating",
            "in_stock",
            "category",
        ]
    ]

    return df.reset_index(drop=True)


# ---------------------------------------------------------
# Database creation
# ---------------------------------------------------------

def create_database(db_path=DB_PATH):
    """
    Create normalized SQLite database.
    """

    connection = sqlite3.connect(db_path)

    cursor = connection.cursor()

    # Enable foreign key enforcement
    cursor.execute("PRAGMA foreign_keys = ON")

    # Drop old tables for reproducibility
    cursor.execute("DROP TABLE IF EXISTS books")
    cursor.execute("DROP TABLE IF EXISTS categories")

    # ---------------------------------
    # Categories table
    # ---------------------------------

    cursor.execute(
        """
        CREATE TABLE categories (
            category_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT NOT NULL UNIQUE
        )
        """
    )

    # ---------------------------------
    # Books table
    # ---------------------------------

    cursor.execute(
        """
        CREATE TABLE books (
            book_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            price_gbp REAL,
            price_inr REAL ,
            rating INTEGER NOT NULL,
            in_stock INTEGER NOT NULL,
            category_id INTEGER NOT NULL,

            FOREIGN KEY (category_id)
                REFERENCES categories(category_id)
        )
        """
    )

    connection.commit()

    return connection


def load_database(df, db_path=DB_PATH):
    """
    Insert cleaned DataFrame into normalized SQLite database.
    """

    connection = create_database(db_path)

    cursor = connection.cursor()

    # ---------------------------------
    # Insert categories
    # ---------------------------------

    categories = (
        df["category"]
        .drop_duplicates()
        .tolist()
    )

    cursor.executemany(
        """
        INSERT INTO categories (category_name)
        VALUES (?)
        """,
        [(category,) for category in categories]
    )

    # Create category lookup
    cursor.execute(
        """
        SELECT category_id, category_name
        FROM categories
        """
    )

    category_lookup = {
        name: category_id
        for category_id, name in cursor.fetchall()
    }

    # ---------------------------------
    # Insert books
    # ---------------------------------

    for _, row in df.iterrows():

        cursor.execute(
            """
            INSERT INTO books (
                title,
                price_gbp,
                price_inr,
                rating,
                in_stock,
                category_id
            )
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                row["title"],
                float(row["price_gbp"]),
                float(row["price_inr"]),
                int(row["rating"]),
                int(row["in_stock"]),
                category_lookup[row["category"]],
            ),
        )

    connection.commit()

    return connection


# ---------------------------------------------------------
# Complete pipeline
# ---------------------------------------------------------

def run_pipeline():

    print("=" * 60)
    print("BOOK DATA PIPELINE")
    print("=" * 60)

    print("\n1. SCRAPING DATA...")

    raw_df = scrape_books(
        min_categories=3,
        minimum_books=60
    )

    print(
        f"\nRaw rows collected: {len(raw_df)}"
    )

    print(
        f"Categories collected: "
        f"{raw_df['category'].nunique()}"
    )

    if len(raw_df) < 60:
        raise ValueError(
            "Pipeline collected fewer than 60 books."
        )

    if raw_df["category"].nunique() < 3:
        raise ValueError(
            "Pipeline collected fewer than 3 categories."
        )

    print("\n2. CLEANING DATA...")

    clean_df = clean_data(raw_df)

    print(
        f"Clean rows: {len(clean_df)}"
    )

    print("\n3. CONVERTING GBP TO INR...")

    print(
        f"Fixed rate: 1 GBP = "
        f"{FIXED_GBP_TO_INR} INR"
    )

    print("\n4. LOADING SQLITE DATABASE...")

    connection = load_database(
        clean_df,
        DB_PATH
    )

    print(
        f"Database created: {DB_PATH}"
    )

    # ---------------------------------
    # Validation
    # ---------------------------------

    cursor = connection.cursor()

    cursor.execute(
        "SELECT COUNT(*) FROM books"
    )

    book_count = cursor.fetchone()[0]

    cursor.execute(
        "SELECT COUNT(*) FROM categories"
    )

    category_count = cursor.fetchone()[0]

    print("\n5. DATABASE VALIDATION")

    print(f"Books: {book_count}")
    print(f"Categories: {category_count}")

    try:
        connection.close()

    except NameError:
        pass

    return clean_df


if __name__ == "__main__":
    run_pipeline()

BOOK DATA PIPELINE

1. SCRAPING DATA...
Found 50 categories.
Scraping category: Travel
  Collected 11 books
Scraping category: Mystery
  Collected 32 books
Scraping category: Historical Fiction
  Collected 26 books

Raw rows collected: 69
Categories collected: 3

2. CLEANING DATA...
Clean rows: 69

3. CONVERTING GBP TO INR...
Fixed rate: 1 GBP = 105.5 INR

4. LOADING SQLITE DATABASE...
Database created: /books.db

5. DATABASE VALIDATION
Books: 69
Categories: 3


/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/tmp/ipykernel_2882/2051923879.py:323: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["price_gbp"] = df["price_gbp"].fillna(
